In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

csv_path = Path("../data/processed/customers_clean.csv")

df = pd.read_csv(csv_path)

connection = sqlite3.connect("../churn_analysis.db")

df.to_sql(
    "customers",
    connection,
    if_exists="replace",
    index=False
)

print("Rows imported:", len(df))
print("Columns:", df.columns.tolist())

Rows imported: 7043
Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [2]:
def run_query(sql):
    return pd.read_sql_query(sql, connection)

In [4]:
run_query("""
SELECT *
FROM customers
LIMIT 10;
""")

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
5,9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,Yes
6,1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,...,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.10,1949.40,No
7,6713-OKOMC,Female,0,No,No,10,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,No,Mailed check,29.75,301.90,No
8,7892-POOKP,Female,0,Yes,No,28,Yes,Yes,Fiber optic,No,...,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes
9,6388-TABGU,Male,0,No,Yes,62,Yes,No,DSL,Yes,...,No,No,No,No,One year,No,Bank transfer (automatic),56.15,3487.95,No


In [5]:
run_query("""
SELECT COUNT(*) AS total_customers
FROM customers;
""")

,total_customers
0,7043


In [6]:
run_query("""
SELECT Churn, COUNT(*) AS customer_count
FROM customers
GROUP BY Churn;
""")

,Churn,customer_count
0,No,5174
1,Yes,1869


In [7]:
run_query("""
SELECT *
FROM customers
WHERE LOWER(TRIM(Churn)) = 'yes'
LIMIT 20;
""")

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
1,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
2,9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,Yes
3,7892-POOKP,Female,0,Yes,No,28,Yes,Yes,Fiber optic,No,...,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes
4,0280-XJGEX,Male,0,No,No,49,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,Yes
5,4190-MFLUW,Female,0,Yes,Yes,10,Yes,No,DSL,No,...,Yes,Yes,No,No,Month-to-month,No,Credit card (automatic),55.20,528.35,Yes
6,8779-QRDMV,Male,1,No,No,1,No,No phone service,DSL,No,...,Yes,No,No,Yes,Month-to-month,Yes,Electronic check,39.65,39.65,Yes
7,1066-JKSGK,Male,0,No,No,1,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.15,20.15,Yes
8,6467-CHFZW,Male,0,Yes,Yes,47,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.35,4749.15,Yes
9,8665-UTDHZ,Male,0,Yes,Yes,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,No,Electronic check,30.20,30.20,Yes


In [8]:
run_query("""
SELECT
    COUNT(*) AS total_customers,
    SUM(
        CASE
            WHEN LOWER(TRIM(Churn)) = 'yes' THEN 1
            ELSE 0
        END
    ) AS churned_customers,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN LOWER(TRIM(Churn)) = 'yes' THEN 1
                ELSE 0
            END
        ) / NULLIF(COUNT(*), 0),
        2
    ) AS churn_rate_percent
FROM customers;
""")

,total_customers,churned_customers,churn_rate_percent
0,7043,1869,26.54


In [9]:
run_query("""
SELECT
    Churn,
    COUNT(*) AS customer_count,
    ROUND(AVG(MonthlyCharges), 2) AS average_monthly_charges
FROM customers
GROUP BY Churn;
""")

,Churn,customer_count,average_monthly_charges
0,No,5174,61.27
1,Yes,1869,74.44


In [10]:
run_query("""
SELECT
    CASE
        WHEN MonthlyCharges < 35 THEN 'Low'
        WHEN MonthlyCharges < 70 THEN 'Medium'
        ELSE 'High'
    END AS charge_segment,
    COUNT(*) AS total_customers
FROM customers
GROUP BY charge_segment
ORDER BY total_customers DESC;
""")

,charge_segment,total_customers
0,High,3591
1,Low,1731
2,Medium,1721


In [11]:
run_query("""
SELECT
    CASE
        WHEN MonthlyCharges < 35 THEN 'Low'
        WHEN MonthlyCharges < 70 THEN 'Medium'
        ELSE 'High'
    END AS charge_segment,

    COUNT(*) AS total_customers,

    SUM(
        CASE
            WHEN LOWER(TRIM(Churn)) = 'yes' THEN 1
            ELSE 0
        END
    ) AS churned_customers,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN LOWER(TRIM(Churn)) = 'yes' THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS churn_rate_percent

FROM customers
GROUP BY charge_segment
ORDER BY churn_rate_percent DESC;
""")

,charge_segment,total_customers,churned_customers,churn_rate_percent
0,High,3591,1274,35.48
1,Medium,1721,407,23.65
2,Low,1731,188,10.86


In [12]:
run_query("""
WITH customer_segments AS (
    SELECT
        customerID,
        Churn,
        CASE
            WHEN MonthlyCharges < 35 THEN 'Low'
            WHEN MonthlyCharges < 70 THEN 'Medium'
            ELSE 'High'
        END AS charge_segment
    FROM customers
)
SELECT
    charge_segment,
    COUNT(*) AS total_customers,
    SUM(
        CASE WHEN LOWER(TRIM(Churn)) = 'yes'
        THEN 1 ELSE 0 END
    ) AS churned_customers
FROM customer_segments
GROUP BY charge_segment;
""")

,charge_segment,total_customers,churned_customers
0,High,3591,1274
1,Low,1731,188
2,Medium,1721,407


In [13]:
run_query("""
WITH segment_rates AS (
    SELECT
        CASE
            WHEN MonthlyCharges < 35 THEN 'Low'
            WHEN MonthlyCharges < 70 THEN 'Medium'
            ELSE 'High'
        END AS charge_segment,
        AVG(
            CASE
                WHEN LOWER(TRIM(Churn)) = 'yes'
                THEN 1.0 ELSE 0.0
            END
        ) * 100 AS churn_rate
    FROM customers
    GROUP BY charge_segment
),
overall_rate AS (
    SELECT
        AVG(
            CASE
                WHEN LOWER(TRIM(Churn)) = 'yes'
                THEN 1.0 ELSE 0.0
            END
        ) * 100 AS churn_rate
    FROM customers
)
SELECT
    s.charge_segment,
    ROUND(s.churn_rate, 2) AS segment_churn_rate,
    ROUND(o.churn_rate, 2) AS overall_churn_rate
FROM segment_rates s
CROSS JOIN overall_rate o
WHERE s.churn_rate > o.churn_rate
ORDER BY s.churn_rate DESC;
""")

,charge_segment,segment_churn_rate,overall_churn_rate
0,High,35.48,26.54


In [14]:
contract_details = pd.DataFrame({
    "Contract": [
        "Month-to-month",
        "One year",
        "Two year"
    ],
    "contract_length": [
        "Monthly",
        "12 months",
        "24 months"
    ]
})

contract_details.to_sql(
    "contract_details",
    connection,
    if_exists="replace",
    index=False
)

run_query("SELECT * FROM contract_details")

,Contract,contract_length
0,Month-to-month,Monthly
1,One year,12 months
2,Two year,24 months


In [15]:
run_query("""
SELECT
    c.customerID,
    c.Contract,
    d.contract_length,
    c.MonthlyCharges,
    c.Churn
FROM customers AS c
INNER JOIN contract_details AS d
    ON c.Contract = d.Contract
LIMIT 10;
""")

,customerID,Contract,contract_length,MonthlyCharges,Churn
0,7590-VHVEG,Month-to-month,Monthly,29.85,No
1,5575-GNVDE,One year,12 months,56.95,No
2,3668-QPYBK,Month-to-month,Monthly,53.85,Yes
3,7795-CFOCW,One year,12 months,42.30,No
4,9237-HQITU,Month-to-month,Monthly,70.70,Yes
5,9305-CDSKC,Month-to-month,Monthly,99.65,Yes
6,1452-KIOVK,Month-to-month,Monthly,89.10,No
7,6713-OKOMC,Month-to-month,Monthly,29.75,No
8,7892-POOKP,Month-to-month,Monthly,104.80,Yes
9,6388-TABGU,One year,12 months,56.15,No


In [16]:
run_query("""
SELECT
    d.contract_length,
    COUNT(*) AS total_customers,
    SUM(
        CASE
            WHEN LOWER(TRIM(c.Churn)) = 'yes'
            THEN 1 ELSE 0
        END
    ) AS churned_customers,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN LOWER(TRIM(c.Churn)) = 'yes'
                THEN 1 ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS churn_rate_percent
FROM customers AS c
INNER JOIN contract_details AS d
    ON c.Contract = d.Contract
GROUP BY d.contract_length
ORDER BY churn_rate_percent DESC;
""")

,contract_length,total_customers,churned_customers,churn_rate_percent
0,Monthly,3875,1655,42.71
1,12 months,1473,166,11.27
2,24 months,1695,48,2.83


In [17]:
run_query("""
SELECT
    c.Contract,
    COUNT(*) AS total_customers,
    COUNT(d.contract_length) AS matched_customers
FROM customers AS c
LEFT JOIN contract_details AS d
    ON c.Contract = d.Contract
GROUP BY c.Contract;
""")

,Contract,total_customers,matched_customers
0,Month-to-month,3875,3875
1,One year,1473,1473
2,Two year,1695,1695


In [18]:
run_query("SELECT COUNT(*) AS total_customers FROM customers")

,total_customers
0,7043


In [19]:
run_query("""
SELECT
    Contract,
    COUNT(*) AS total_customers,
    SUM(
        CASE
            WHEN LOWER(TRIM(Churn)) = 'yes' THEN 1
            ELSE 0
        END
    ) AS churned_customers,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN LOWER(TRIM(Churn)) = 'yes' THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS churn_rate
FROM customers
GROUP BY Contract
ORDER BY churn_rate DESC;
""")

,Contract,total_customers,churned_customers,churn_rate
0,Month-to-month,3875,1655,42.71
1,One year,1473,166,11.27
2,Two year,1695,48,2.83


In [20]:
run_query("""
WITH contract_stats AS (
    SELECT
        Contract,
        COUNT(*) AS total_customers,
        AVG(
            CASE
                WHEN LOWER(TRIM(Churn)) = 'yes' THEN 1.0
                ELSE 0.0
            END
        ) * 100 AS churn_rate
    FROM customers
    GROUP BY Contract
)
SELECT
    Contract,
    total_customers,
    ROUND(churn_rate, 2) AS churn_rate,
    RANK() OVER (ORDER BY churn_rate DESC) AS churn_rank
FROM contract_stats
ORDER BY churn_rank;
""")

,Contract,total_customers,churn_rate,churn_rank
0,Month-to-month,3875,42.71,1
1,One year,1473,11.27,2
2,Two year,1695,2.83,3


In [21]:
run_query("""
WITH contract_stats AS (
    SELECT
        Contract,
        AVG(
            CASE
                WHEN LOWER(TRIM(Churn)) = 'yes' THEN 1.0
                ELSE 0.0
            END
        ) * 100 AS churn_rate
    FROM customers
    GROUP BY Contract
)
SELECT
    Contract,
    ROUND(churn_rate, 2) AS contract_churn_rate,
    ROUND(AVG(churn_rate) OVER (), 2) AS average_of_contract_rates
FROM contract_stats
ORDER BY contract_churn_rate DESC;
""")

,Contract,contract_churn_rate,average_of_contract_rates
0,Month-to-month,42.71,18.94
1,One year,11.27,18.94
2,Two year,2.83,18.94


In [22]:
run_query("""
SELECT
    Contract,
    ROUND(
        AVG(
            CASE WHEN LOWER(TRIM(Churn)) = 'yes'
                 THEN 100.0 ELSE 0.0 END
        ), 2
    ) AS contract_churn_rate,
    ROUND(
        (SELECT AVG(
            CASE WHEN LOWER(TRIM(Churn)) = 'yes'
                 THEN 100.0 ELSE 0.0 END
        ) FROM customers), 2
    ) AS overall_churn_rate
FROM customers
GROUP BY Contract
ORDER BY contract_churn_rate DESC;
""")

,Contract,contract_churn_rate,overall_churn_rate
0,Month-to-month,42.71,26.54
1,One year,11.27,26.54
2,Two year,2.83,26.54


In [23]:
run_query("""
SELECT
    Contract,
    Churn,
    MonthlyCharges,
    ROUND(
        AVG(MonthlyCharges) OVER (PARTITION BY Contract),
        2
    ) AS avg_charge_for_contract
FROM customers
LIMIT 15;
""")

,Contract,Churn,MonthlyCharges,avg_charge_for_contract
0,Month-to-month,No,29.85,66.4
1,Month-to-month,Yes,53.85,66.4
2,Month-to-month,Yes,70.70,66.4
3,Month-to-month,Yes,99.65,66.4
4,Month-to-month,No,89.10,66.4
5,Month-to-month,No,29.75,66.4
6,Month-to-month,Yes,104.80,66.4
7,Month-to-month,No,49.95,66.4
8,Month-to-month,Yes,103.70,66.4
9,Month-to-month,No,105.50,66.4
